In [1]:
! pip install datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("knkarthick/dialogsum")

c:\Users\kaila\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kaila\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kaila\.cache\huggingface\hub\datasets--knkarthick--dialogsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see thi

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [4]:
ds['train'][1]['dialogue']

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [5]:
ds['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

In [6]:
!pip install transformers

In [7]:
# WITHOUT FINE TUNING

In [8]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("summarization", model="facebook/bart-large-cnn")

c:\Users\kaila\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kaila\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download.

In [9]:
article_1=ds['train'][1]['dialogue']

In [10]:
pipe(article_1,max_length=20,min_length=10,do_sample=False)

[{'summary_text': 'Ricky has received his Polio, Tetanus and Hepatitis B shots.'}]

In [11]:
ds['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

In [12]:
# WITH FINE TUNING

In [13]:
# Load Model Directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")


In [14]:
def preprocess_function(batch):
    source = batch["dialogue"]
    target = batch["summary"]

    source_ids = tokenizer(source, truncation=True, padding='max_length', max_length=128)
    target_ids = tokenizer(target, truncation=True, padding='max_length', max_length=128)

    labels = target_ids["input_ids"]
    labels = [[(label if label != tokenizer.pad_token_id else -100) for label in labels_example]
              for labels_example in labels]

    return {
        "input_ids": source_ids["input_ids"],
        "attention_mask": source_ids["attention_mask"],
        "labels": labels
    }


In [15]:
df_source = ds.map(preprocess_function, batched=True)


Map: 100%|██████████| 1500/1500 [00:00<00:00, 2646.31 examples/s]


In [16]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size=8,
    num_train_epochs=2,
    remove_unused_columns=True
)


In [18]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = df_source['train'],
    eval_dataset=df_source['test']
)


In [2]:
from datasets import load_dataset
from transformers import pipeline
import evaluate

# 1) Load your dataset
ds = load_dataset("knkarthick/dialogsum")

# 2) Load summarization model (without fine-tuning)
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# 3) Load ROUGE metric
rouge = evaluate.load("rouge")

# 4) Take small sample for fast evaluation (CPU friendly)
sample_size = 10
test_data = ds["test"].select(range(sample_size))

predictions = []
references = []

for item in test_data:
    dialogue = item["dialogue"]
    ref_summary = item["summary"]

    # Generate summary
    pred = summarizer(
        dialogue,
        max_length=60,
        min_length=20,
        do_sample=False
    )[0]["summary_text"]

    predictions.append(pred)
    references.append(ref_summary)

# 5) Calculate ROUGE
results = rouge.compute(predictions=predictions, references=references)

print("✅ ROUGE Results on DialogSum (sample =", sample_size, ")")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


Device set to use cpu


✅ ROUGE Results on DialogSum (sample = 10 )
rouge1: 0.2639
rouge2: 0.0787
rougeL: 0.1900
rougeLsum: 0.1898
